# DeepNoise: CNN Training Pipeline (Native PyTorch)

This notebook loads the 2D Mel-spectrogram features, calculates class weights to balance the loss function against class skew, splits the dataset (70/15/15), and trains our custom 2D Convolutional Neural Network (CNN) in native PyTorch with GPU acceleration.

In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import TensorDataset, DataLoader

# Ensure VRAM allocation environment variables are set before PyTorch initializes
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Append parent and local paths
sys.path.append(os.path.dirname(os.path.abspath('.')))
sys.path.append(os.path.abspath('.'))
from config import FEATURES_DIR, MODELS_DIR, RESULTS_DIR, BATCH_SIZE, EPOCHS, LEARNING_RATE, DEVICE
from cnn_model import build_cnn_model

print("CUDA available for PyTorch:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Target GPU:", torch.cuda.get_device_name(0))
print("Using device:", DEVICE)

## 1. Load 2D Spectrogram Features

We load the Mel-spectrogram `.npy` arrays, preserving their 2D dimensions `(128, 173)`. We then add a single channel dimension to represent them as grayscale images: `(128, 173, 1)`.

In [ ]:
def load_dataset_2d():
    X = []
    y = []
    
    # Identify directories representing classes
    class_folders = sorted([d for d in os.listdir(FEATURES_DIR) if os.path.isdir(os.path.join(FEATURES_DIR, d))])
    
    # Map each class name to a unique integer index
    class_to_label = {class_name: idx for idx, class_name in enumerate(class_folders)}
    print("Class-to-Label mapping:", class_to_label)
    
    for class_name in class_folders:
        class_dir = os.path.join(FEATURES_DIR, class_name)
        npy_files = glob.glob(os.path.join(class_dir, "*.npy"))
        label = class_to_label[class_name]
        
        print(f"Loading {len(npy_files)} files for class: {class_name}")
        for file_path in npy_files:
            mel_spec = np.load(file_path)
            X.append(mel_spec)
            y.append(label)
            
    X = np.array(X)
    y = np.array(y)
    
    # Expand dimensions to add the channels axis: (N, 128, 173) -> (N, 128, 173, 1)
    X = np.expand_dims(X, axis=-1)
    
    return X, y, class_to_label

X, y, class_to_label = load_dataset_2d()
print(f"\nDataset loaded! Features shape: {X.shape} | Labels shape: {y.shape}")

## 2. Compute Class Weights

Because the dataset is highly unbalanced, we compute class weights to balance the loss function.

In [ ]:
# Compute class weights inversely proportional to class frequencies
unique_classes = np.unique(y)
weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=y
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Computed Class Weights:")
for class_name, idx in class_to_label.items():
    print(f"  Class '{class_name}' ({idx}): weight = {weights[idx]:.4f}")

## 3. Train / Validation / Test Split

We split the dataset (70% train, 15% validation, 15% test) and wrap them in PyTorch DataLoader objects.

In [ ]:
# Step 1: Split into 70% Train and 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step 2: Split Temporary into equal Validation and Test sets (15% and 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Wrap datasets in PyTorch DataLoader
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train set size:      {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size:       {X_test.shape[0]}")

## 4. Initialize the PyTorch CNN Model

We instantiate our native PyTorch custom 2D CNN architecture, transfer it to the target device, and define the loss function and optimizer.

In [ ]:
# Instantiate and target model to device
model = build_cnn_model(num_classes=4)
model.to(DEVICE)

# Set optimizer and weighted loss function
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(model)

## 5. Train the Model with Custom Loop (with Early Stopping)

We implement the training loop in native PyTorch, checking validation loss at the end of each epoch to trigger early stopping if the validation loss stops improving.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
best_model_path = os.path.join(MODELS_DIR, "best_cnn.pth")

best_val_loss = float("inf")
patience = 5
patience_counter = 0
best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}

history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

print(f"Training CNN model on {DEVICE}...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = loss_fn(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(x_batch)
        preds = torch.argmax(outputs, dim=-1)
        train_correct += torch.sum(preds == y_batch).item()
        train_total += len(x_batch)
        
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val = x_val.to(DEVICE)
            y_val = y_val.to(DEVICE)
            
            outputs = model(x_val)
            loss = loss_fn(outputs, y_val)
            
            val_loss += loss.item() * len(x_val)
            preds = torch.argmax(outputs, dim=-1)
            val_correct += torch.sum(preds == y_val).item()
            val_total += len(x_val)
            
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_state_dict, best_model_path)
        print(f"  -> Saved new best model checkpoint to {best_model_path}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  -> Early stopping triggered at epoch {epoch+1}.")
            break

# Restore best weights
model.load_state_dict(best_state_dict)
torch.save(best_state_dict, best_model_path)
print("Training complete! Best weights loaded and saved.")

## 6. Plot Learning History

We plot the training and validation loss and accuracy.

In [ ]:
plt.figure(figsize=(14, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history["loss"], label="Train Loss", color="royalblue")
plt.plot(history["val_loss"], label="Val Loss", color="tomato")
plt.title("CNN: Training vs. Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history["accuracy"], label="Train Acc", color="royalblue")
plt.plot(history["val_accuracy"], label="Val Acc", color="tomato")
plt.title("CNN: Training vs. Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
os.makedirs(RESULTS_DIR, exist_ok=True)
history_path = os.path.join(RESULTS_DIR, "cnn_training_history.png")
plt.savefig(history_path, dpi=300)
print(f"Saved training curves to: {history_path}")
plt.show()